## NB_lab_test_trends
Builds **`<catalog>.gold.lab_test_trends`** as defined in `Gold/sttm_silver_to_gold.xlsx` (sheet `lab_test_trends`).

* **Grain:** one row per test type (`test_name`) per calendar month
* **Driving table:** `<catalog>.silver.lab_results` grouped by `test_name`, year and month of `test_date`
* **Joins (LEFT, optional):** `silver.calendar` on `date_key = yyyyMMdd(first day of month)` for `month_name`, `quarter_name`
* **Window:** `LAG(abnormal_rate_pct)` partitioned by `test_name`, ordered by `year`, `month`
* **Filters:** none
* **Load type:** FULL, `CREATE OR REPLACE TABLE ... AS SELECT`

In [0]:
# Catalog placeholder from the STTM (<catalog>); defaults to adb_caresync
dbutils.widgets.text("catalog", "adb_caresync")
catalog = dbutils.widgets.get("catalog").strip()

source_schema = f"{catalog}.silver"
target_table = f"{catalog}.gold.lab_test_trends"

print("=" * 80)
print("[STEP 1] Notebook parameters")
print(f"  catalog       : {catalog}")
print(f"  source schema : {source_schema}")
print(f"  target table  : {target_table}")
print("=" * 80)

In [0]:
# monthly_agg: one row per test_name per month; with_lag adds the previous month's abnormal rate
lab_test_trends_sql = f"""
CREATE OR REPLACE TABLE {target_table} AS
WITH monthly_agg AS (
  SELECT
    test_name,
    year(test_date)                                         AS year,
    month(test_date)                                        AS month,
    COUNT(*)                                                AS total_tests,
    COUNT(CASE WHEN result_status = 'Abnormal' THEN 1 END)  AS abnormal_test_count,
    COUNT(CASE WHEN result_status = 'Critical' THEN 1 END)  AS critical_test_count
  FROM {source_schema}.lab_results
  GROUP BY test_name, year(test_date), month(test_date)
),
with_rates AS (
  SELECT
    m.*,
    CAST(ROUND(m.abnormal_test_count * 100.0 / m.total_tests, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(ROUND(m.critical_test_count * 100.0 / m.total_tests, 2) AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly_agg m
),
with_lag AS (
  SELECT
    r.*,
    LAG(r.abnormal_rate_pct) OVER (PARTITION BY r.test_name ORDER BY r.year, r.month) AS prev_month_abnormal_rate_pct
  FROM with_rates r
)
SELECT
  CAST(w.test_name AS STRING)                                  AS test_name,
  CAST(w.year AS INT)                                          AS year,
  CAST(w.month AS INT)                                         AS month,
  CAST(c.month_name AS STRING)                                 AS month_name,
  CAST(c.quarter_name AS STRING)                               AS quarter_name,
  CAST(w.total_tests AS INT)                                   AS total_tests,
  CAST(w.abnormal_test_count AS INT)                           AS abnormal_test_count,
  CAST(w.critical_test_count AS INT)                           AS critical_test_count,
  w.abnormal_rate_pct                                          AS abnormal_rate_pct,
  w.critical_rate_pct                                          AS critical_rate_pct,
  CAST(w.prev_month_abnormal_rate_pct AS DECIMAL(5,2))         AS prev_month_abnormal_rate_pct,
  CAST(
    CASE
      WHEN w.prev_month_abnormal_rate_pct IS NULL THEN NULL
      ELSE w.abnormal_rate_pct - w.prev_month_abnormal_rate_pct
    END AS DECIMAL(6,2))                                       AS abnormal_rate_change_pct
FROM with_lag w
LEFT JOIN {source_schema}.calendar c
  ON c.date_key = CAST(date_format(make_date(w.year, w.month, 1), 'yyyyMMdd') AS INT)
"""

print(f"[STEP 2] Building {target_table}")
print("  Query prepared:")
print(lab_test_trends_sql)
spark.sql(lab_test_trends_sql)
print(f"  {target_table} created successfully")
print("=" * 80)

In [0]:
# Validate grain (one row per test per month) and that every lab test is counted once
validation_sql = f"""
SELECT
  (SELECT COUNT(DISTINCT test_name, year(test_date), month(test_date)) FROM {source_schema}.lab_results) AS source_test_months,
  COUNT(*)                                                  AS target_rows,
  COUNT(DISTINCT test_name, year, month)                    AS distinct_test_months,
  SUM(total_tests)                                          AS total_tests,
  (SELECT COUNT(*) FROM {source_schema}.lab_results)        AS source_lab_results,
  COUNT(CASE WHEN month_name IS NULL THEN 1 END)            AS missing_calendar_match
FROM {target_table}
"""
print("[STEP 3] Validating")
print("  Query prepared:")
print(validation_sql)
v = spark.sql(validation_sql).first()
for k, val in v.asDict().items():
    print(f"  {k:<24}: {val}")

if not (v.source_test_months == v.target_rows == v.distinct_test_months):
    raise ValueError("Grain check failed: expected one row per test_name per month")
if v.total_tests != v.source_lab_results:
    raise ValueError("Reconciliation failed: total_tests does not match silver.lab_results row count")
print("  Grain and reconciliation checks passed")
print("=" * 80)

display(spark.table(target_table).orderBy("test_name", "year", "month").limit(15))

In [0]:
# Return the gold table record count to the caller (ADF / parent notebook)
record_count = spark.table(target_table).count()
print(f"[STEP 4] Record count for {target_table}: {record_count}")
dbutils.notebook.exit(str(record_count))